# 运行时反馈循环

看不到真实命令输出的agent在猜。一个反馈运行器捕捉stdout，stderr，返回值以及计时信息，填入结构化的记录中，这样下一轮就能够读取。此时agent能够基于事实做出反应而非它对事实的预测。

## 问题描述

agent先说“现在开始跑测试”。下一条信息继续说“所有测试通过”。事实却是没有测试执行。agent想象了输出，或者它跑了命令但是没有去读结果，或者它读了结果却偷偷截掉了那行失败。

一个反馈运行器解决这个问题。所有的命令都经过这个运行器。每条记录包含命令，捕获的stdout和stderr，返回值，墙钟时间，以及一行agent笔记。agent在下一轮可以读取记录。验证门禁在任务结束后读取这些记录。

## 基本概念
```mermaid
flowchart TD
A[Agent Loop]-->B[run_with_feedback.py]-->C[subprocess]-->D[stdout/stderr/exit/duration]-->E[feedback_record.jsonl]-->F[Verification Gate]
E-->A
```

### 反馈记录里有什么
|字段|为什么重要|
|---|---|
|command|准确的argv，没有shell展开的以外|
|stdout_tail|最后的N行，事实性截取|
|stderr_tail|最后的N行，与stdout分离|
|exit_code|无可争议的成功信号|
|duration_ms|暴露慢探针和失控进程|
|started_at|用于重放的时间戳|
|agent_note|agent写的一行，关于它预期什么|

### 截断是确定性的

阶段的原因是防止日志暴涨，而且agent能够看见头尾重要的内容（最终的错误或者总结）。

### 反馈和遥测

遥测是给人类运维跨时间审查运行使用的。反馈是给这次运行的下一轮用的。它们共享字段，但住在不同的文件里，有不同的保留策略。

### 没有反馈就拒绝推进

如果运行器在捕获退出之前出错，记录就带`exit_code:null`和`error:<reason>`。agent循环必须拒绝在`null`推出上声称成功。没有退出，就不往前推进。

### 碎碎念

1. 写入时脱敏，不是读取事。 任何碰stdout和stderr的记录都可能泄露密钥。
2. 轮转策略，而非单个文件。 防止单个`feedback_record.jsonl`过长，使用多个文件轮转。
3. 给重试链加父命令ID。  

# 开始编码

对应本章核心：**命令必须经反馈运行器**、**结构化 `feedback_record.jsonl`（stdout/stderr/exit/duration/note）**、**确定性截断 + 写入脱敏 + 轮转**、**`exit_code=null` 拒推进**、**重试 `parent_id`**。  
玩具先跑通记录与门禁；生产段用 **LangChain 工具 + DeepSeek** 强制读反馈再宣称成功。不硬凑 PyTorch。


## 1. 教学玩具：FeedbackRunner + jsonl 存储

- `subprocess` 捕获真实输出；写入前 `redact`；尾部截断。
- 按条数轮转多文件；`claim_success` 在 null/非 0 时抛错。


In [ ]:
from __future__ import annotations

import json
import re
import shutil
import sys
import subprocess
import tempfile
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Sequence


SECRET_PATTERNS = [
    re.compile(r"(?i)(api[_-]?key|token|secret|password)\s*[=:]\s*([^\s'\"]+)"),
    re.compile(r"(?i)(Bearer\s+)([A-Za-z0-9._\-]+)"),
]


def redact(text: str) -> str:
    """
    写入时脱敏，避免密钥进反馈文件。

    Args:
        text: 原始 stdout/stderr。
    Returns:
        cleaned: 脱敏后文本。
    """
    out = text
    for pat in SECRET_PATTERNS:
        out = pat.sub(lambda m: f"{m.group(1)}***", out)
    return out


def truncate_tail(text: str, *, max_lines: int = 40, max_chars: int = 4000) -> str:
    """
    确定性截断：保留尾部（错误/总结常在末尾）。

    Args:
        text: 全文。
        max_lines: 最多保留行数。
        max_chars: 最多字符。
    Returns:
        tail: 截断后的文本。
    """
    lines = text.splitlines()
    if len(lines) > max_lines:
        lines = lines[-max_lines:]
        text = "\n".join(lines)
        prefix = f"[truncated {len(text.splitlines())}+ lines kept as tail]\n"
        text = prefix + text
    else:
        text = "\n".join(lines)
    if len(text) > max_chars:
        text = text[-max_chars:]
        text = "[truncated chars]\n" + text
    return text


@dataclass
class FeedbackRecord:
    """单次命令的结构化反馈（给下一轮 agent / 验证门禁读）。"""

    id: str
    command: list[str]
    stdout_tail: str
    stderr_tail: str
    exit_code: int | None
    duration_ms: int
    started_at: float
    agent_note: str = ""
    error: str | None = None
    parent_id: str | None = None

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @property
    def ok(self) -> bool:
        return self.exit_code == 0

    def claim_success(self) -> None:
        """
        没有确切 exit 就不许声称成功。

        Raises:
            RuntimeError: exit_code 为 null 或非 0。
        """
        if self.exit_code is None:
            raise RuntimeError(f"refuse advance: exit_code is null (error={self.error})")
        if self.exit_code != 0:
            raise RuntimeError(f"refuse advance: exit_code={self.exit_code}")


class FeedbackStore:
    """jsonl 反馈存储 + 简单按条数轮转（非单文件无限涨）。"""

    def __init__(self, root: Path, *, max_records_per_file: int = 20) -> None:
        """
        Args:
            root: 反馈目录。
            max_records_per_file: 单文件最大记录数，超出轮转。
        """
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_records_per_file = max_records_per_file
        self._index = 0
        self._count_in_file = 0
        self._current = self.root / f"feedback_record_{self._index:03d}.jsonl"

    @property
    def current_path(self) -> Path:
        return self._current

    def append(self, record: FeedbackRecord) -> Path:
        """
        Args:
            record: 反馈记录。
        Returns:
            path: 写入的 jsonl 路径。
        """
        if self._count_in_file >= self.max_records_per_file:
            self._index += 1
            self._count_in_file = 0
            self._current = self.root / f"feedback_record_{self._index:03d}.jsonl"
        with self._current.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record.to_dict(), ensure_ascii=False) + "\n")
        self._count_in_file += 1
        return self._current

    def read_all(self) -> list[dict[str, Any]]:
        """
        Returns:
            rows: 按文件名顺序的全部记录。
        """
        rows: list[dict[str, Any]] = []
        for p in sorted(self.root.glob("feedback_record_*.jsonl")):
            for line in p.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rows.append(json.loads(line))
        return rows

    def latest(self) -> dict[str, Any] | None:
        rows = self.read_all()
        return rows[-1] if rows else None


class FeedbackRunner:
    """所有命令经此运行器：subprocess → 结构化记录 → 下一轮可读。"""

    def __init__(self, store: FeedbackStore, *, timeout_s: float = 30.0) -> None:
        self.store = store
        self.timeout_s = timeout_s

    def run(
        self,
        command: Sequence[str],
        *,
        agent_note: str = "",
        parent_id: str | None = None,
        cwd: Path | None = None,
        env: dict[str, str] | None = None,
    ) -> FeedbackRecord:
        """
        Args:
            command: argv 列表（不经 shell 展开）。
            agent_note: agent 对预期结果的一行说明。
            parent_id: 重试链父命令 id。
            cwd: 工作目录。
            env: 额外环境变量。
        Returns:
            record: 已落盘的反馈记录。
        """
        cmd = [str(x) for x in command]
        rid = uuid.uuid4().hex[:12]
        started = time.time()
        exit_code: int | None = None
        error: str | None = None
        stdout = ""
        stderr = ""
        try:
            proc = subprocess.run(
                cmd,
                cwd=str(cwd) if cwd else None,
                capture_output=True,
                text=True,
                timeout=self.timeout_s,
                env=env,
                shell=False,
            )
            exit_code = int(proc.returncode)
            stdout = proc.stdout or ""
            stderr = proc.stderr or ""
        except subprocess.TimeoutExpired as e:
            error = f"timeout after {self.timeout_s}s"
            stdout = (e.stdout or "") if isinstance(e.stdout, str) else ""
            stderr = (e.stderr or "") if isinstance(e.stderr, str) else ""
            exit_code = None
        except OSError as e:
            error = f"os_error: {e}"
            exit_code = None

        duration_ms = int((time.time() - started) * 1000)
        record = FeedbackRecord(
            id=rid,
            command=cmd,
            stdout_tail=truncate_tail(redact(stdout)),
            stderr_tail=truncate_tail(redact(stderr)),
            exit_code=exit_code,
            duration_ms=duration_ms,
            started_at=started,
            agent_note=agent_note,
            error=error,
            parent_id=parent_id,
        )
        self.store.append(record)
        return record


def make_feedback_root() -> Path:
    """
    Returns:
        root: 临时反馈目录。
    """
    return Path(tempfile.mkdtemp(prefix="feedback_"))


def py_cmd(*code_or_args: str) -> list[str]:
    """
    Args:
        code_or_args: 若单段字符串则作为 ``-c`` 代码；否则作为额外 argv。
    Returns:
        argv: 使用当前解释器的 argv。
    """
    if len(code_or_args) == 1 and not code_or_args[0].startswith("-"):
        return [sys.executable, "-c", code_or_args[0]]
    return [sys.executable, *code_or_args]


print("runtime feedback ready | runner + jsonl + redact + rotation")


## 2. 玩具示例

真跑命令、失败拒推进、脱敏、轮转、null exit、parent_id 重试链。


In [ ]:
def demo_runtime_feedback() -> None:
    """真实命令入账、脱敏、轮转、null exit 拒推进、重试链 parent_id。"""
    root = make_feedback_root()
    try:
        store = FeedbackStore(root, max_records_per_file=3)
        runner = FeedbackRunner(store, timeout_s=5.0)

        # 成功命令：必须真跑，不能空想
        r1 = runner.run(py_cmd("print('all tests passed')"), agent_note="expect exit 0")
        assert r1.exit_code == 0
        assert "all tests passed" in r1.stdout_tail
        r1.claim_success()
        print("real stdout captured ok")

        # 失败命令
        r2 = runner.run(py_cmd("import sys; print('boom', file=sys.stderr); sys.exit(2)"), agent_note="expect fail")
        assert r2.exit_code == 2
        assert "boom" in r2.stderr_tail
        try:
            r2.claim_success()
            raise AssertionError("should refuse")
        except RuntimeError as e:
            assert "exit_code=2" in str(e)
            print("non-zero refuse advance ok")

        # 脱敏：写入时打码
        r3 = runner.run(
            py_cmd("print('api_key=sk-live-SECRET123'); print('Bearer tok_ABC')"),
            agent_note="should redact",
        )
        assert "sk-live-SECRET123" not in r3.stdout_tail
        assert "***" in r3.stdout_tail
        print("redact on write ok")

        # 轮转：第 4 条进新文件（max=3）
        r4 = runner.run(py_cmd("print(4)"), agent_note="rotate")
        files = sorted(root.glob("feedback_record_*.jsonl"))
        assert len(files) >= 2, files
        assert r4.id in json.dumps(store.read_all())
        print("rotation ok")

        # 捕获前出错 → exit_code null → 拒推进
        r5 = runner.run(["/nonexistent/bin/nope"], agent_note="missing binary")
        assert r5.exit_code is None and r5.error
        try:
            r5.claim_success()
            raise AssertionError("null exit should block")
        except RuntimeError as e:
            assert "null" in str(e)
            print("null exit refuse advance ok")

        # 重试链 parent_id
        parent = runner.run(py_cmd("import sys; sys.exit(1)"), agent_note="first try")
        child = runner.run(
            py_cmd("print('retry ok')"),
            agent_note="retry after fail",
            parent_id=parent.id,
        )
        assert child.parent_id == parent.id and child.exit_code == 0
        print("retry parent_id ok")

        latest = store.latest()
        assert latest and latest["id"] == child.id
        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_runtime_feedback()


## 3. 生产级：LangChain 反馈工具 + DeepSeek

工具：`run_with_feedback`（argv 白名单）/ `read_latest_feedback` / `claim_success`。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_STORE: FeedbackStore | None = None
PROD_RUNNER: FeedbackRunner | None = None


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_feedback() -> str:
    """
    Returns:
        json: 反馈根目录。
    """
    global PROD_ROOT, PROD_STORE, PROD_RUNNER
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = make_feedback_root()
    PROD_STORE = FeedbackStore(PROD_ROOT, max_records_per_file=10)
    PROD_RUNNER = FeedbackRunner(PROD_STORE, timeout_s=10.0)
    return json.dumps({"root": str(PROD_ROOT)}, ensure_ascii=False)


def _runner() -> FeedbackRunner:
    if PROD_RUNNER is None:
        reset_feedback()
    assert PROD_RUNNER is not None
    return PROD_RUNNER


def _store() -> FeedbackStore:
    if PROD_STORE is None:
        reset_feedback()
    assert PROD_STORE is not None
    return PROD_STORE


class EmptyArgs(BaseModel):
    pass


class RunArgs(BaseModel):
    argv_json: str = Field(description='JSON 数组 argv，如 ["python","-c","print(1)"]')
    agent_note: str = Field("", description="预期一行说明")
    parent_id: str | None = Field(None, description="重试时填父记录 id")


class ClaimArgs(BaseModel):
    record_id: str = Field(description="要声称成功的反馈记录 id")


def build_feedback_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        return reset_feedback()

    def _run(**kwargs: Any) -> str:
        a = RunArgs(**kwargs)
        try:
            argv = json.loads(a.argv_json)
        except json.JSONDecodeError as e:
            return json.dumps({"ok": False, "error": f"bad argv_json: {e}"}, ensure_ascii=False)
        if not isinstance(argv, list) or not all(isinstance(x, str) for x in argv):
            return json.dumps({"ok": False, "error": "argv must be list[str]"}, ensure_ascii=False)
        # 白名单：只允许 python / echo 演示，避免任意 shell
        allowed = {"python", "python3", "echo", Path(sys.executable).name, sys.executable}
        if not argv or argv[0] not in allowed:
            return json.dumps({"ok": False, "error": "argv[0] not in allowlist"}, ensure_ascii=False)
        rec = _runner().run(argv, agent_note=a.agent_note, parent_id=a.parent_id)
        return json.dumps(rec.to_dict(), ensure_ascii=False)

    def _latest(**kwargs: Any) -> str:
        row = _store().latest()
        return json.dumps(row or {"empty": True}, ensure_ascii=False)

    def _history(**kwargs: Any) -> str:
        rows = _store().read_all()
        return json.dumps({"count": len(rows), "records": rows[-8:]}, ensure_ascii=False)

    def _claim(**kwargs: Any) -> str:
        rid = ClaimArgs(**kwargs).record_id
        rows = {r["id"]: r for r in _store().read_all()}
        if rid not in rows:
            return json.dumps({"ok": False, "error": "unknown record"}, ensure_ascii=False)
        rec = FeedbackRecord(**rows[rid])
        try:
            rec.claim_success()
            return json.dumps({"ok": True, "advanced": True, "record_id": rid}, ensure_ascii=False)
        except RuntimeError as e:
            return json.dumps({"ok": False, "advanced": False, "error": str(e)}, ensure_ascii=False)

    return [
        StructuredTool.from_function(name="reset_feedback", description="重建反馈目录。", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(
            name="run_with_feedback",
            description="经反馈运行器执行 argv（白名单），写入 jsonl。",
            func=_run,
            args_schema=RunArgs,
        ),
        StructuredTool.from_function(name="read_latest_feedback", description="读最新一条反馈。", func=_latest, args_schema=EmptyArgs),
        StructuredTool.from_function(name="read_feedback_history", description="读最近反馈历史。", func=_history, args_schema=EmptyArgs),
        StructuredTool.from_function(
            name="claim_success",
            description="仅当记录 exit_code==0 时允许推进；null/非0 拒绝。",
            func=_claim,
            args_schema=ClaimArgs,
        ),
    ]


FEEDBACK_TOOLS = build_feedback_tools()


def build_feedback_agent():
    """
    Returns:
        agent: 必须经反馈运行器读事实的 control agent。
    """
    system = (
        "你是反馈循环代理。禁止空想测试结果；必须 run_with_feedback，再 read_latest_feedback，"
        "只有 claim_success 成功才能说通过。失败可带 parent_id 重试。\n"
        "argv_json 用 JSON 数组字符串。用中文简短说明。"
    )
    return create_agent(get_llm(), FEEDBACK_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 28) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"runtime feedback production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证拒推进。


In [ ]:
def demo_prod_feedback() -> None:
    """脚本化门禁 +（有 key 时）DeepSeek。"""
    reset_feedback()
    runner = _runner()

    ok = runner.run(py_cmd("print('green')"), agent_note="unit ok")
    ok.claim_success()

    bad = runner.run(py_cmd("import sys; sys.exit(1)"), agent_note="should fail")
    try:
        bad.claim_success()
        raise AssertionError("must refuse")
    except RuntimeError:
        pass

    missing = runner.run(["/no/such/cmd"], agent_note="null exit")
    assert missing.exit_code is None
    print("=== scripted ===")
    print(json.dumps({"ok_advance": True, "fail_refused": True, "null_refused": True}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_feedback_agent()
    prompt = (
        "reset_feedback，用 run_with_feedback 跑 "
        "argv_json=[\"python3\",\"-c\",\"import sys; sys.exit(1)\"] agent_note=先失败，"
        "read_latest_feedback，尝试 claim_success（应失败），"
        "再带 parent_id 重试 argv_json=[\"python3\",\"-c\",\"print('recovered')\"]，"
        "claim_success，中文说明为何不能空想测试通过。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    rows = _store().read_all()
    assert len(rows) >= 2
    # 最后一条应成功且可能带 parent
    last = rows[-1]
    assert last.get("exit_code") == 0
    print("PROD DEMO OK")


demo_prod_feedback()
